# 🏦 Financial Fraud Detection Pipeline
## Notebook 2: Preprocessing & Feature Engineering

**Goal:** Clean the data, scale features, handle severe class imbalance using SMOTE, and export a train/test split ready for modeling.

**Steps:**
1. Load raw data
2. Scale `Amount` and `Time`
3. Train/test split (stratified)
4. Apply SMOTE to training set only
5. Validate & save processed data

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✅')

## 2. Load Raw Data

In [ ]:
df = pd.read_csv('../data/creditcard.csv')

print(f'Shape: {df.shape}')
print(f'Fraud cases: {df["Class"].sum():,} ({df["Class"].mean()*100:.4f}%)')
print(f'Legitimate cases: {(df["Class"]==0).sum():,}')
df.head(3)

## 3. Scale `Amount` and `Time`

`V1–V28` are already PCA-transformed (zero mean, unit variance).  
`Amount` and `Time` are raw and need to be standardized to avoid dominating the model.

In [ ]:
scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

# Drop original unscaled columns
df_clean = df.drop(columns=['Amount', 'Time'])

print('Scaling complete ✅')
print(f'Amount_scaled — mean: {df_clean["Amount_scaled"].mean():.4f}, std: {df_clean["Amount_scaled"].std():.4f}')
print(f'Time_scaled   — mean: {df_clean["Time_scaled"].mean():.4f},   std: {df_clean["Time_scaled"].std():.4f}')
df_clean.head(3)

## 4. Feature Matrix & Target Split

In [ ]:
X = df_clean.drop(columns=['Class'])
y = df_clean['Class']

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'Feature columns: {X.columns.tolist()}')

## 5. Stratified Train / Test Split

Using `stratify=y` ensures both splits maintain the same fraud ratio (~0.17%).  
We apply SMOTE **only on the training set** — never on test data, to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set:   {X_train.shape[0]:,} rows | Fraud: {y_train.sum():,} ({y_train.mean()*100:.4f}%)')
print(f'Test set:       {X_test.shape[0]:,} rows  | Fraud: {y_test.sum():,} ({y_test.mean()*100:.4f}%)')
print('\nStratification preserved ✅')

## 6. Handle Class Imbalance with SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic fraud samples by interpolating between existing fraud cases — much better than simple duplication.

⚠️ Applied to **training data only** to prevent leakage into evaluation.

In [ ]:
print('Before SMOTE:')
print(f'  Legitimate: {(y_train==0).sum():,}')
print(f'  Fraud:      {(y_train==1).sum():,}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(f'  Legitimate: {(y_train_res==0).sum():,}')
print(f'  Fraud:      {(y_train_res==1).sum():,}')
print(f'\nNew training set shape: {X_train_res.shape}')
print('SMOTE applied ✅')

## 7. Visualize Class Balance Before vs After SMOTE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

before = pd.Series(y_train).value_counts().sort_index()
after  = pd.Series(y_train_res).value_counts().sort_index()

for ax, counts, title in zip(axes, [before, after], ['Before SMOTE', 'After SMOTE']):
    bars = ax.bar(['Legitimate', 'Fraud'], counts.values,
                  color=['#4C72B0', '#DD8452'], edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', fontsize=10)

plt.suptitle('Class Distribution: Before vs After SMOTE (Training Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/smote_comparison.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/smote_comparison.png ✅')

## 8. Save Processed Data

In [ ]:
# Combine into DataFrames for saving
train_resampled = pd.DataFrame(X_train_res, columns=X.columns)
train_resampled['Class'] = y_train_res.values

test_df = pd.DataFrame(X_test, columns=X.columns)
test_df['Class'] = y_test.values

# Save
train_resampled.to_csv('../data/train_resampled.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

print('Files saved:')
print(f'  data/train_resampled.csv — {train_resampled.shape[0]:,} rows')
print(f'  data/test.csv           — {test_df.shape[0]:,} rows')
print('\nPreprocessing complete ✅ Ready for modeling!')

## 9. Preprocessing Summary

| Step | Action | Result |
|---|---|---|
| **Scaling** | StandardScaler on `Amount` & `Time` | Zero mean, unit variance |
| **Split** | 80/20 stratified train/test | Fraud ratio preserved in both sets |
| **SMOTE** | Synthetic oversampling on train only | Balanced 50/50 training set |
| **Leakage prevention** | SMOTE never touches test set | Evaluation remains unbiased |
| **Output** | `train_resampled.csv`, `test.csv` | Ready for `03_modeling.ipynb` |

### ➡️ Next Step: `03_modeling.ipynb` — train Random Forest & XGBoost, evaluate with ROC-AUC, confusion matrix, and export predictions for Tableau